In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [4]:
# Load data from markdown table format
df = pd.read_excel(r"C:\Users\user\Downloads\Groups 1 & 7.xlsx")

# Parse the markdown table
#df = pd.read_csv(StringIO(data), sep='|', skiprows=2, header=0, skipinitialspace=True)
#df = df.drop(df.columns[[0, -1]], axis=1)  # Remove empty first/last columns
#df.columns = [col.strip() for col in df.columns]
df

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,No,32,Venmo,Weekly
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,No,41,Bank Transfer,Bi-Weekly
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,No,24,Venmo,Quarterly
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,No,24,Venmo,Weekly


In [5]:
# Data cleaning
df['Age'] = df['Age'].astype(int)
df['Purchase Amount (USD)'] = pd.to_numeric(df['Purchase Amount (USD)'])
df['Previous Purchases'] = pd.to_numeric(df['Previous Purchases'])

In [6]:
# ====================================================
# 1. Data Visualization
# ====================================================
plt.figure(figsize=(12, 18))
plt.suptitle('Retail Sales Analysis', fontsize=16)

# Top 10 Selling Categories
plt.subplot(3, 2, 1)
top_categories = df['Category'].value_counts().nlargest(10)
sns.barplot(x=top_categories.values, y=top_categories.index, palette='viridis')
plt.title('Top 10 Selling Categories')
plt.xlabel('Units Sold')

# Revenue Distribution by Category
plt.subplot(3, 2, 2)
revenue_by_category = df.groupby('Category')['Purchase Amount (USD)'].sum().nlargest(10)
sns.barplot(x=revenue_by_category.values, y=revenue_by_category.index, palette='magma')
plt.title('Top 10 Revenue Generating Categories')
plt.xlabel('Total Revenue (USD)')

# Sales Distribution by Season
plt.subplot(3, 2, 3)
season_dist = df['Season'].value_counts()
plt.pie(season_dist, labels=season_dist.index, autopct='%1.1f%%', 
        colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'])
plt.title('Sales Distribution by Season')

# Purchase Amount Distribution
plt.subplot(3, 2, 4)
sns.histplot(df['Purchase Amount (USD)'], bins=20, kde=True, color='skyblue')
plt.title('Purchase Amount Distribution')
plt.xlabel('Purchase Amount (USD)')

# Review Ratings Distribution
plt.subplot(3, 2, 5)
sns.boxplot(x=df['Review Rating'], color='lightgreen')
plt.title('Review Ratings Distribution')

# Top 10 States by Sales
plt.subplot(3, 2, 6)
top_states = df['Location'].value_counts().nlargest(10)
sns.barplot(x=top_states.values, y=top_states.index, palette='rocket')
plt.title('Top 10 States by Sales Volume')
plt.xlabel('Units Sold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('retail_visualizations.png', dpi=300)
plt.close()

C:\Users\user\AppData\Local\Temp\ipykernel_10872\2177916019.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_categories.values, y=top_categories.index, palette='viridis')
C:\Users\user\AppData\Local\Temp\ipykernel_10872\2177916019.py:17: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=revenue_by_category.values, y=revenue_by_category.index, palette='magma')
C:\Users\user\AppData\Local\Temp\ipykernel_10872\2177916019.py:42: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_states.values, y=top_states.index, palette='rocket')


In [8]:
# ====================================================
# 2. Inventory Level Metrics
# ====================================================
inventory_metrics = df.groupby('Category').agg(
    Total_Sold=('Item Purchased', 'count'),
    Avg_Selling_Price=('Purchase Amount (USD)', 'mean'),
    Stock_Turnover_Rate=('Item Purchased', lambda x: x.count()/x.nunique()),
    Revenue=('Purchase Amount (USD)', 'sum')
).reset_index()

inventory_metrics['Demand_Variability'] = df.groupby('Category')['Purchase Amount (USD)'].std() / inventory_metrics['Avg_Selling_Price']
inventory_metrics = inventory_metrics.sort_values('Total_Sold', ascending=False)

# Save to CSV
inventory_metrics.to_csv('inventory_metrics.csv', index=False)

In [9]:
# ====================================================
# 3. Sales Demand Forecasting
# ====================================================
# Create time series data (using index as pseudo-time)
ts_data = df.groupby('Customer ID')['Purchase Amount (USD)'].sum().sort_index()

# Split into train/test (last 20% as test)
split_idx = int(len(ts_data) * 0.8)
train = ts_data[:split_idx]
test = ts_data[split_idx:]

# Holt-Winters forecasting model
model = ExponentialSmoothing(train, seasonal='add', seasonal_periods=12)
model_fit = model.fit()
forecast = model_fit.forecast(len(test))

# Create forecast dataframe
forecast_df = pd.DataFrame({
    'Customer_ID': test.index,
    'Actual_Sales': test.values,
    'Forecasted_Sales': forecast.values
})

# Calculate accuracy metrics
mape = np.mean(np.abs((test - forecast) / test)) * 100
accuracy = 100 - mape

# Save forecast results
forecast_df.to_csv('demand_forecast.csv', index=False)

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [10]:
# ====================================================
# 4. Profitability Analysis
# ====================================================
# Calculate profitability metrics (assume 30% profit margin)
profit_df = df.groupby('Item Purchased').agg(
    Units_Sold=('Item Purchased', 'count'),
    Total_Revenue=('Purchase Amount (USD)', 'sum'),
    Avg_Price=('Purchase Amount (USD)', 'mean')
).reset_index()

profit_df['Estimated_Profit'] = profit_df['Total_Revenue'] * 0.3
profit_df['Profit_per_Unit'] = profit_df['Estimated_Profit'] / profit_df['Units_Sold']
profit_df = profit_df.sort_values('Estimated_Profit', ascending=False)

# Save to CSV
profit_df.to_csv('profitability_analysis.csv', index=False)

# ====================================================
# Generate Summary Report
# ====================================================
report = f"""
Retail Performance Analysis Report
=================================

Key Metrics:
- Total Records: {len(df):,}
- Total Revenue: ${df['Purchase Amount (USD)'].sum():,.2f}
- Average Transaction Value: ${df['Purchase Amount (USD)'].mean():.2f}
- Most Popular Category: {inventory_metrics.iloc[0]['Category']}
- Top Selling Item: {profit_df.iloc[0]['Item Purchased']}
- Forecast Accuracy: {accuracy:.2f}%

Inventory Insights:
1. Highest turnover category: {inventory_metrics.iloc[0]['Category']} 
   (Turnover rate: {inventory_metrics.iloc[0]['Stock_Turnover_Rate']:.2f})
2. Most stable demand: {inventory_metrics.sort_values('Demand_Variability').iloc[0]['Category']}
   (Variability: {inventory_metrics.sort_values('Demand_Variability').iloc[0]['Demand_Variability']:.3f})
3. Highest revenue category: {revenue_by_category.idxmax()}
   (Revenue: ${revenue_by_category.max():,.2f})

Profitability Highlights:
- Most profitable item: {profit_df.iloc[0]['Item Purchased']}
  (Total profit: ${profit_df.iloc[0]['Estimated_Profit']:,.2f})
- Highest margin item: {profit_df.sort_values('Profit_per_Unit', ascending=False).iloc[0]['Item Purchased']}
  (Profit per unit: ${profit_df.sort_values('Profit_per_Unit', ascending=False).iloc[0]['Profit_per_Unit']:.2f})

Recommendations:
1. Increase stock for high-turnover categories: {', '.join(top_categories.index[:3])}
2. Focus marketing on seasonal items for {df['Season'].mode()[0]} season
3. Develop promotions for low-margin high-volume items
"""

with open('retail_analysis_report.txt', 'w') as f:
    f.write(report)

print("Analysis complete! Output files generated:")
print("- retail_visualizations.png")
print("- inventory_metrics.csv")
print("- demand_forecast.csv")
print("- profitability_analysis.csv")
print("- retail_analysis_report.txt")

Analysis complete! Output files generated:
- retail_visualizations.png
- inventory_metrics.csv
- demand_forecast.csv
- profitability_analysis.csv
- retail_analysis_report.txt
